# 面试题：如何从零实现 Conformer 编码器与 CTC，并正确隔离 padding？

## 面试回答主线

Conformer 用 macaron 两段半权重 FFN、全局 self-attention 和局部卷积共同编码语音帧；CTC 则在不知道帧级对齐时，对所有合法 blank/重复路径做动态规划求和。变长 batch 的 padding 合同必须贯穿整个网络：输入投影后清零，attention 同时屏蔽 key 与无效 query，每个 residual 后重新清零，卷积前后也要清零。只在最终 logits 上 mask 不够，因为 depthwise convolution 会让非零 padding 污染最后一个有效帧。

下面不调用 `nn.Transformer`、`nn.MultiheadAttention`、`nn.Conv1d` 或 `CTCLoss`，而是用参数矩阵、`einsum`、`F.conv1d` 和 `logsumexp` 手写结构、前向、CTC DP 与训练。

## 真实案例：六类中文智能家居命令识别

训练集包含“开灯、关灯、开窗、关窗、开门、关门”各 4 段带噪声声学序列，共 24 条；测试集为 6 条新噪声录音。每条录音具有真实长度 5–7，batch 补齐到 7 帧。声学特征是可读的受控原型，不是实际麦克风数据，因此只验证变长序列、Conformer 和 CTC 机制。

In [1]:
import math  # 导入平方根用于缩放点积注意力。
import torch  # 导入 PyTorch 以生成声学序列并执行真实反向传播。
from torch import nn  # 导入基础模块和可学习参数。
import torch.nn.functional as F  # 导入底层卷积和激活函数。
torch.set_num_threads(1)  # 固定小型语音实验使用单线程。
torch.manual_seed(71)  # 固定声学噪声、参数初始化与训练结果。
TOKEN_TO_ID = {"<blank>": 0, "开": 1, "关": 2, "灯": 3, "窗": 4, "门": 5}  # 定义 CTC blank 与五个汉字 token。
ID_TO_TOKEN = {value: key for key, value in TOKEN_TO_ID.items()}  # 创建预测编号到汉字的反向表。
COMMANDS = [("开灯", [1, 3]), ("关灯", [2, 3]), ("开窗", [1, 4]), ("关窗", [2, 4]), ("开门", [1, 5]), ("关门", [2, 5])]  # 定义六类命令和目标 token。
PROTOTYPES = torch.tensor([  # 定义 blank 与五个汉字的六维声学原型。
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00],  # blank 或短暂停顿原型。
    [1.00, 0.10, 0.05, 0.20, 0.00, 0.10],  # “开”的受控声学原型。
    [0.10, 1.00, 0.05, 0.20, 0.00, 0.10],  # “关”的受控声学原型。
    [0.05, 0.10, 1.00, 0.15, 0.05, 0.20],  # “灯”的受控声学原型。
    [0.05, 0.10, 0.10, 1.00, 0.20, 0.05],  # “窗”的受控声学原型。
    [0.05, 0.10, 0.10, 0.20, 1.00, 0.05],  # “门”的受控声学原型。
], dtype=torch.float32)  # 完成六维声学原型定义。
TRAIN_PATTERNS = [  # 定义四种训练帧级发音与停顿模式。
    ["A", "A", "B", "B", "_"],  # 五帧连续双字后停顿。
    ["_", "A", "A", "_", "B", "B"],  # 六帧前后含停顿。
    ["A", "A", "A", "_", "B", "B"],  # 六帧首字持续更长。
    ["A", "_", "A", "B", "_", "B", "B"],  # 七帧含字内短暂停顿。
]  # 完成训练时长模式定义。
TEST_PATTERNS = [  # 为六条测试录音定义新的时长组合。
    ["_", "A", "A", "B", "B"],  # 五帧前导停顿模式。
    ["A", "A", "_", "B", "B", "_"],  # 六帧中间与尾部停顿模式。
    ["A", "_", "A", "_", "B", "B", "_"],  # 七帧双停顿模式。
    ["_", "A", "A", "A", "B", "B"],  # 六帧首字长发音模式。
    ["A", "A", "B", "_", "B"],  # 五帧次字被停顿分隔。
    ["A", "_", "A", "B", "B", "B", "_"],  # 七帧次字长发音模式。
]  # 完成测试变长模式定义。
def synthesize_utterance(target_ids, pattern, seed):  # 从 token 原型和时长模式生成一段带噪声录音。
    generator = torch.Generator().manual_seed(seed)  # 为当前录音建立独立噪声源。
    frames = []  # 收集逐帧声学向量。
    for symbol in pattern:  # 按可读发音模式生成每一帧。
        token_id = 0 if symbol == "_" else target_ids[0] if symbol == "A" else target_ids[1]  # 把模式符号映射到 blank 或命令字符。
        noise_scale = 0.035 if token_id == 0 else 0.075  # 为停顿和发音使用不同噪声幅度。
        frame = PROTOTYPES[token_id] + torch.randn(6, generator=generator) * noise_scale  # 生成带独立扰动的六维观测。
        frames.append(frame)  # 保存当前声学帧。
    return torch.stack(frames)  # 返回真实长度的声学序列。
train_utterances = []  # 收集二十四条训练录音。
train_targets = []  # 收集对应两字 CTC 目标。
train_names = []  # 收集可读训练样本名称。
for command_index, (command, target_ids) in enumerate(COMMANDS):  # 遍历六个中文命令。
    for variant_index, pattern in enumerate(TRAIN_PATTERNS):  # 为每个命令生成四种时长变体。
        train_utterances.append(synthesize_utterance(target_ids, pattern, 1000 + command_index * 10 + variant_index))  # 生成当前训练录音。
        train_targets.append(torch.tensor(target_ids, dtype=torch.long))  # 保存两字目标而不提供帧级对齐。
        train_names.append(f"{command}-训练{variant_index + 1}")  # 保存可追踪样本名称。
test_utterances = []  # 收集六条留出测试录音。
test_targets = []  # 收集六条测试命令目标。
test_names = []  # 收集可读测试命令名称。
for command_index, ((command, target_ids), pattern) in enumerate(zip(COMMANDS, TEST_PATTERNS)):  # 对齐命令和各自新测试模式。
    test_utterances.append(synthesize_utterance(target_ids, pattern, 2000 + command_index))  # 使用全新噪声种子生成测试录音。
    test_targets.append(torch.tensor(target_ids, dtype=torch.long))  # 保存测试两字目标。
    test_names.append(command)  # 保存命令文本供结果展示。
def pad_utterances(utterances):  # 把变长录音补齐成 batch 并生成有效帧 mask。
    lengths = torch.tensor([utterance.shape[0] for utterance in utterances], dtype=torch.long)  # 记录每条录音真实长度。
    maximum_length = int(lengths.max())  # 读取当前 batch 最大帧数。
    padded = torch.zeros(len(utterances), maximum_length, utterances[0].shape[1])  # 创建零 padding 声学张量。
    valid_mask = torch.zeros(len(utterances), maximum_length, dtype=torch.bool)  # 创建逐帧有效性 mask。
    for index, utterance in enumerate(utterances):  # 遍历每条变长录音。
        padded[index, :utterance.shape[0]] = utterance  # 把真实帧复制到 batch 前部。
        valid_mask[index, :utterance.shape[0]] = True  # 标记真实帧且保持 padding 为假。
    return padded, lengths, valid_mask  # 返回补齐输入、真实长度和布尔 mask。
train_padded, train_lengths, train_valid_mask = pad_utterances(train_utterances)  # 补齐二十四条训练录音。
test_padded, test_lengths, test_valid_mask = pad_utterances(test_utterances)  # 补齐六条测试录音。
print("训练/测试 batch 形状：", tuple(train_padded.shape), tuple(test_padded.shape))  # 展示真实批量声学输入尺寸。
print("测试命令与真实长度：", list(zip(test_names, test_lengths.tolist())))  # 展示六条变长录音而非裸 token。
print("测试样本1前三帧声学特征：", [[round(float(value), 3) for value in frame] for frame in test_padded[0, :3]])  # 展示模型真正读取的连续观测。

训练/测试 batch 形状： (24, 7, 6) (6, 7, 6)
测试命令与真实长度： [('开灯', 5), ('关灯', 6), ('开窗', 7), ('关窗', 6), ('开门', 5), ('关门', 7)]
测试样本1前三帧声学特征： [[-0.018, 0.01, -0.008, 0.015, 0.008, -0.047], [0.963, 0.104, 0.062, 0.253, -0.054, 0.086], [0.915, 0.038, 0.097, 0.277, -0.086, 0.128]]


## Baseline（基线）：逐帧最近声学原型 + CTC collapse

基线不知道命令类别，只把每个有效帧分配给最近的固定声学原型，再执行“先合并连续重复、后删除 blank”的 CTC greedy collapse。它没有上下文建模，但在干净教学数据上可能已经很强，因此后续重点比较可训练 loss 和 padding 合同，而不是强行宣称深层模型一定胜出。

In [2]:
def ctc_collapse(path):  # 按 CTC 规则合并连续重复并移除 blank。
    collapsed = []  # 收集合并重复后的非 blank token。
    previous = None  # 保存上一帧原始 token 以判断连续重复。
    for token_id in path:  # 顺序扫描逐帧预测路径。
        if token_id != previous and token_id != 0:  # 只保留变化后的非 blank token。
            collapsed.append(token_id)  # 保存当前新字符。
        previous = token_id  # 更新上一帧原始预测。
    return collapsed  # 返回无帧级重复的字符序列。
def token_ids_to_text(token_ids):  # 把预测编号列表还原为中文字符串。
    return "".join(ID_TO_TOKEN[token_id] for token_id in token_ids)  # 拼接全部非 blank 汉字。
def prototype_baseline(padded, lengths):  # 对变长 batch 执行最近声学原型识别。
    decoded = []  # 收集每条录音最终文本。
    paths = []  # 收集逐帧最近原型路径。
    for sample_index, length in enumerate(lengths.tolist()):  # 遍历录音并尊重真实长度。
        valid_frames = padded[sample_index, :length]  # 截取当前录音真实帧。
        distances = ((valid_frames[:, None, :] - PROTOTYPES[None, :, :]) ** 2).sum(dim=-1)  # 计算每帧到六个原型的平方距离。
        path = distances.argmin(dim=-1).tolist()  # 选择每帧最近原型作为贪心路径。
        paths.append(path)  # 保存可审计帧路径。
        decoded.append(token_ids_to_text(ctc_collapse(path)))  # 应用 CTC collapse 得到命令文本。
    return decoded, paths  # 返回逐条文本和帧级路径。
baseline_decoded, baseline_paths = prototype_baseline(test_padded, test_lengths)  # 在六条测试录音上执行固定原型基线。
baseline_exact = sum(prediction == gold for prediction, gold in zip(baseline_decoded, test_names)) / len(test_names)  # 计算整句完全匹配率。
print("命令  真实长度  最近原型帧路径       collapse结果  正确")  # 输出逐句基线账本表头。
for name, length, path, prediction in zip(test_names, test_lengths.tolist(), baseline_paths, baseline_decoded):  # 对齐命令、长度、路径和文本。
    print(f"{name}      {length}      {path!s:<22}  {prediction:<4}    {prediction == name}")  # 展示重复与 blank 如何被折叠。
print(f"最近原型 Baseline 整句 exact match={baseline_exact:.1%}")  # 汇总可比较基线指标。

命令  真实长度  最近原型帧路径       collapse结果  正确
开灯      5      [0, 1, 1, 3, 3]         开灯      True
关灯      6      [2, 2, 0, 3, 3, 0]      关灯      True
开窗      7      [1, 0, 1, 0, 4, 4, 0]   开开窗     False
关窗      6      [0, 2, 2, 2, 4, 4]      关窗      True
开门      5      [1, 1, 5, 0, 5]         开门门     False
关门      7      [2, 0, 2, 5, 5, 5, 0]   关关门     False
最近原型 Baseline 整句 exact match=50.0%


## 核心实现一：逐层传播 padding mask 的手写 Conformer

`apply_valid_mask` 不只在最终输出调用：输入投影、两段 FFN residual、attention residual、卷积输入、卷积输出和最终 LayerNorm 后都会清零无效帧。attention 还同时屏蔽无效 key 和 query。卷积前清零最关键，因为 kernel=3 会读取相邻帧；卷积后再次清零则防止有效邻居把 padding 位置重新写成非零。

In [3]:
def apply_valid_mask(values, valid_mask):  # 把无效帧严格重置为零。
    return values * valid_mask.unsqueeze(-1).to(values.dtype)  # 广播逐帧布尔 mask 到隐藏维。
class ManualFeedForward(nn.Module):  # 定义 Conformer macaron 前馈子层。
    def __init__(self, model_dim, expansion_dim):  # 初始化升维和降维参数。
        super().__init__()  # 注册基础模块状态。
        self.input_weight = nn.Parameter(torch.randn(model_dim, expansion_dim) / math.sqrt(model_dim))  # 创建隐藏维到扩展维投影。
        self.input_bias = nn.Parameter(torch.zeros(expansion_dim))  # 创建扩展层偏置。
        self.output_weight = nn.Parameter(torch.randn(expansion_dim, model_dim) / math.sqrt(expansion_dim))  # 创建扩展维回隐藏维投影。
        self.output_bias = nn.Parameter(torch.zeros(model_dim))  # 创建输出偏置。
    def forward(self, values):  # 执行两层前馈网络。
        expanded = F.silu(values @ self.input_weight + self.input_bias)  # 使用 SiLU 激活扩展帧表示。
        return expanded @ self.output_weight + self.output_bias  # 投影回模型隐藏维。
class ManualSelfAttention(nn.Module):  # 定义单头缩放点积自注意力。
    def __init__(self, model_dim):  # 初始化 Q、K、V 和输出参数。
        super().__init__()  # 注册基础模块状态。
        self.model_dim = model_dim  # 保存隐藏维用于缩放分数。
        self.query_weight = nn.Parameter(torch.randn(model_dim, model_dim) / math.sqrt(model_dim))  # 创建 query 投影。
        self.key_weight = nn.Parameter(torch.randn(model_dim, model_dim) / math.sqrt(model_dim))  # 创建 key 投影。
        self.value_weight = nn.Parameter(torch.randn(model_dim, model_dim) / math.sqrt(model_dim))  # 创建 value 投影。
        self.output_weight = nn.Parameter(torch.randn(model_dim, model_dim) / math.sqrt(model_dim))  # 创建 attention 输出投影。
    def forward(self, values, valid_mask, strict_padding=True):  # 执行 key mask 并可选择严格 query mask。
        queries = values @ self.query_weight  # 计算逐帧 query。
        keys = values @ self.key_weight  # 计算逐帧 key。
        projected_values = values @ self.value_weight  # 计算逐帧 value。
        scores = torch.einsum("btd,bsd->bts", queries, keys) / math.sqrt(self.model_dim)  # 手写所有帧对的缩放点积。
        scores = scores.masked_fill(~valid_mask[:, None, :], -1e4)  # 永远禁止有效 query 读取 padding key。
        attention = torch.softmax(scores, dim=-1)  # 沿来源帧维归一化注意力。
        if strict_padding:  # 正确实现还要让 padding query 不产生任何路由。
            attention = attention * valid_mask[:, :, None].to(attention.dtype)  # 把无效 query 的整行权重清零。
        context = torch.einsum("bts,bsd->btd", attention, projected_values)  # 聚合全局时序上下文。
        output = context @ self.output_weight  # 投影 attention 上下文。
        return apply_valid_mask(output, valid_mask) if strict_padding else output, attention  # 严格模式再次清零无效 query 输出。
class ManualConvolution(nn.Module):  # 定义 GLU、depthwise conv 与 pointwise 输出。
    def __init__(self, model_dim, kernel_size=3):  # 初始化手写卷积模块参数。
        super().__init__()  # 注册基础模块状态。
        self.kernel_size = kernel_size  # 保存局部卷积核宽度。
        self.glu_weight = nn.Parameter(torch.randn(model_dim, model_dim * 2) / math.sqrt(model_dim))  # 创建 GLU 两分支投影。
        self.glu_bias = nn.Parameter(torch.zeros(model_dim * 2))  # 创建 GLU 偏置。
        self.depthwise_weight = nn.Parameter(torch.randn(model_dim, 1, kernel_size) * 0.12)  # 创建每通道独立卷积核。
        self.depthwise_bias = nn.Parameter(torch.zeros(model_dim))  # 创建 depthwise 卷积偏置。
        self.output_weight = nn.Parameter(torch.randn(model_dim, model_dim) / math.sqrt(model_dim))  # 创建 pointwise 输出投影。
        self.output_bias = nn.Parameter(torch.zeros(model_dim))  # 创建卷积模块输出偏置。
    def forward(self, values, valid_mask, strict_padding=True):  # 执行局部卷积并控制 padding 边界。
        convolution_input = apply_valid_mask(values, valid_mask) if strict_padding else values  # 正确路径在卷积前强制 padding 为零。
        glu_values = convolution_input @ self.glu_weight + self.glu_bias  # 计算 GLU 内容与门控两分支。
        content, gate = glu_values.chunk(2, dim=-1)  # 沿隐藏维拆分 GLU 两部分。
        gated = content * torch.sigmoid(gate)  # 用门控调制局部卷积输入。
        channel_first = gated.transpose(1, 2)  # 转换为批量、通道、时间布局。
        convolved = F.conv1d(channel_first, self.depthwise_weight, self.depthwise_bias, padding=self.kernel_size // 2, groups=channel_first.shape[1])  # 用底层 conv1d 执行逐通道局部卷积。
        activated = F.silu(convolved.transpose(1, 2))  # 转回时间布局并应用 SiLU。
        output = activated @ self.output_weight + self.output_bias  # 用 pointwise 矩阵混合通道。
        return apply_valid_mask(output, valid_mask) if strict_padding else output  # 正确路径在卷积后再次清除 padding。
class ManualConformerBlock(nn.Module):  # 定义一个完整 macaron Conformer block。
    def __init__(self, model_dim=24, expansion_dim=48):  # 初始化归一化、双 FFN、attention 与 convolution。
        super().__init__()  # 注册基础模块状态。
        self.norm_ffn1 = nn.LayerNorm(model_dim)  # 创建第一段 FFN 前归一化。
        self.norm_attention = nn.LayerNorm(model_dim)  # 创建 attention 前归一化。
        self.norm_convolution = nn.LayerNorm(model_dim)  # 创建卷积前归一化。
        self.norm_ffn2 = nn.LayerNorm(model_dim)  # 创建第二段 FFN 前归一化。
        self.final_norm = nn.LayerNorm(model_dim)  # 创建 block 最终归一化。
        self.ffn1 = ManualFeedForward(model_dim, expansion_dim)  # 创建第一段 macaron FFN。
        self.attention = ManualSelfAttention(model_dim)  # 创建全局自注意力。
        self.convolution = ManualConvolution(model_dim)  # 创建局部 depthwise 卷积模块。
        self.ffn2 = ManualFeedForward(model_dim, expansion_dim)  # 创建第二段 macaron FFN。
    def forward(self, values, valid_mask, strict_padding=True):  # 执行并在每个 residual 边界传播 mask。
        hidden = apply_valid_mask(values, valid_mask) if strict_padding else values  # 正确路径首先清除输入 padding。
        first_ffn = self.ffn1(self.norm_ffn1(hidden))  # 计算第一段前馈残差分支。
        hidden = hidden + 0.5 * first_ffn  # 按 macaron 结构加入半权重 FFN。
        hidden = apply_valid_mask(hidden, valid_mask) if strict_padding else hidden  # 在第一段 residual 后再次清零 padding。
        attention_output, attention = self.attention(self.norm_attention(hidden), valid_mask, strict_padding)  # 计算带 key/query 合同的注意力。
        hidden = hidden + attention_output  # 加入全局上下文 residual。
        hidden = apply_valid_mask(hidden, valid_mask) if strict_padding else hidden  # 在 attention residual 后重置 padding。
        convolution_input = self.norm_convolution(hidden)  # 对局部卷积分支执行前归一化。
        convolution_output = self.convolution(convolution_input, valid_mask, strict_padding)  # 在卷积前后应用严格 mask。
        hidden = hidden + convolution_output  # 加入局部卷积 residual。
        hidden = apply_valid_mask(hidden, valid_mask) if strict_padding else hidden  # 在卷积 residual 后重置 padding。
        second_ffn = self.ffn2(self.norm_ffn2(hidden))  # 计算第二段前馈残差分支。
        hidden = hidden + 0.5 * second_ffn  # 加入第二个半权重 FFN。
        hidden = apply_valid_mask(hidden, valid_mask) if strict_padding else hidden  # 在第二段 residual 后重置 padding。
        hidden = self.final_norm(hidden)  # 对完整 block 输出执行归一化。
        hidden = apply_valid_mask(hidden, valid_mask) if strict_padding else hidden  # 在 block 出口保证无效帧严格为零。
        return hidden, attention  # 返回编码表示和注意力矩阵。
class ManualConformerCTC(nn.Module):  # 定义声学投影、Conformer block 与 CTC 分类头。
    def __init__(self, input_dim=6, model_dim=24, vocabulary_size=6):  # 初始化输入与输出参数。
        super().__init__()  # 注册基础模块状态。
        self.input_weight = nn.Parameter(torch.randn(input_dim, model_dim) * 0.20)  # 创建声学输入投影。
        self.input_bias = nn.Parameter(torch.zeros(model_dim))  # 创建输入投影偏置。
        self.block = ManualConformerBlock(model_dim, model_dim * 2)  # 创建一个手写 Conformer block。
        self.output_weight = nn.Parameter(torch.randn(model_dim, vocabulary_size) * 0.18)  # 创建逐帧 CTC 分类头。
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))  # 创建词表输出偏置。
    def forward(self, acoustic_features, valid_mask, strict_padding=True):  # 执行严格或故障对照前向。
        hidden = acoustic_features @ self.input_weight + self.input_bias  # 把六维声学观测映射到隐藏维。
        hidden = apply_valid_mask(hidden, valid_mask) if strict_padding else hidden  # 正确路径在输入投影后立刻清零 padding。
        hidden, attention = self.block(hidden, valid_mask, strict_padding)  # 执行全局与局部时序编码。
        logits = hidden @ self.output_weight + self.output_bias  # 输出每帧六个 CTC token 分数。
        logits = apply_valid_mask(logits, valid_mask) if strict_padding else logits  # 正确路径也清除最终 padding logits。
        return logits, attention, hidden  # 返回逐帧 logits、attention 与编码表示。
torch.manual_seed(73)  # 固定 Conformer 参数初始化。
model = ManualConformerCTC()  # 创建手写 Conformer-CTC 模型。
preview_logits, preview_attention, preview_hidden = model(test_padded[:2], test_valid_mask[:2])  # 对两条变长录音执行未训练前向。
print("输入/logits/attention/hidden 形状：", tuple(test_padded[:2].shape), tuple(preview_logits.shape), tuple(preview_attention.shape), tuple(preview_hidden.shape))  # 展示完整时序张量路径。
print("短样本 padding hidden 最大绝对值：", round(float(preview_hidden[0, test_lengths[0]:].abs().max()), 8))  # 直接观察 block 出口 padding 已为零。
print("短样本有效 query 的 attention 行和：", [round(float(value), 5) for value in preview_attention[0, :test_lengths[0]].sum(dim=-1)])  # 检查有效 query 概率质量。

输入/logits/attention/hidden 形状： (2, 7, 6) (2, 7, 6) (2, 7, 7) (2, 7, 24)
短样本 padding hidden 最大绝对值： 0.0
短样本有效 query 的 attention 行和： [1.0, 1.0, 1.0, 1.0, 1.0]


## 核心实现二：手写 CTC 前向动态规划

目标 `[开, 灯]` 扩展为 `[blank, 开, blank, 灯, blank]`。每个状态可以从原状态、前一状态转移；当前符号不是 blank 且不等于前两个状态时，还可跳两格。递推在 log 空间使用 `logsumexp`，每条样本只遍历 `input_length`，因此 DP 不读取补齐帧。

In [4]:
def single_ctc_negative_log_likelihood(log_probabilities, target, input_length):  # 为一条录音手写 CTC 前向算法。
    extended = [0]  # 从开头 blank 建立扩展目标。
    for token_id in target.tolist():  # 逐个插入真实字符及间隔 blank。
        extended.append(token_id)  # 添加当前真实字符。
        extended.append(0)  # 在字符后添加 blank 状态。
    initial_states = []  # 收集第一帧到各扩展状态的 log 概率。
    for state_index, token_id in enumerate(extended):  # 初始化扩展目标全部状态。
        if state_index <= 1:  # 第一帧只能停在开头 blank 或第一个字符。
            initial_states.append(log_probabilities[0, token_id])  # 读取合法起始状态发射概率。
        else:  # 其余状态第一帧不可达。
            initial_states.append(log_probabilities.new_tensor(-1e9))  # 用极小 log 概率表示不可达。
    previous = torch.stack(initial_states)  # 形成第一帧动态规划向量。
    for time_index in range(1, int(input_length)):  # 只遍历真实有效声学帧。
        current_states = []  # 收集当前帧每个扩展状态值。
        for state_index, token_id in enumerate(extended):  # 遍历扩展目标状态。
            candidates = [previous[state_index]]  # 允许从相同状态保持。
            if state_index > 0:  # 非首状态可从前一状态前进。
                candidates.append(previous[state_index - 1])  # 添加一步前进来源。
            can_skip = state_index > 1 and token_id != 0 and token_id != extended[state_index - 2]  # 判断能否跳过中间 blank。
            if can_skip:  # 仅不同的连续字符允许跨两格。
                candidates.append(previous[state_index - 2])  # 添加两步跳转来源。
            transition = torch.logsumexp(torch.stack(candidates), dim=0)  # 对全部合法路径概率求和。
            current_states.append(transition + log_probabilities[time_index, token_id])  # 加上当前状态的发射 log 概率。
        previous = torch.stack(current_states)  # 推进到下一真实帧。
    terminal = torch.logsumexp(previous[-2:], dim=0)  # CTC 可在末字符或末尾 blank 结束。
    return -terminal  # 返回当前录音负 log-likelihood。
def manual_ctc_loss(logits, targets, lengths):  # 汇总一个 batch 的手写 CTC loss。
    log_probabilities = torch.log_softmax(logits, dim=-1)  # 计算逐帧词表 log 概率。
    losses = []  # 收集逐条录音负 log-likelihood。
    for sample_index, target in enumerate(targets):  # 遍历变长目标序列。
        losses.append(single_ctc_negative_log_likelihood(log_probabilities[sample_index], target, lengths[sample_index]))  # 在真实输入长度内运行 DP。
    return torch.stack(losses).mean(), torch.stack(losses)  # 返回 batch 均值和逐样本损失。
initial_ctc_loss, initial_sample_losses = manual_ctc_loss(preview_logits, test_targets[:2], test_lengths[:2])  # 对两条样本计算未训练 CTC 目标。
example_extended_target = [0, int(test_targets[0][0]), 0, int(test_targets[0][1]), 0]  # 构造第一条命令的扩展状态序列。
print("命令", test_names[0], "的扩展目标：", [ID_TO_TOKEN[token_id] for token_id in example_extended_target])  # 展示 blank 插入后的 DP 状态。
print("首两条未训练 CTC NLL：", [round(float(value), 4) for value in initial_sample_losses])  # 展示真实动态规划数值而非只看 shape。
print("首两条平均 CTC loss：", round(float(initial_ctc_loss), 4))  # 汇总未训练损失起点。

命令 开灯 的扩展目标： ['<blank>', '开', '<blank>', '灯', '<blank>']
首两条未训练 CTC NLL： [5.0605, 9.5057]
首两条平均 CTC loss： 7.2831


## 真实训练与逐句结果

训练时始终启用严格 padding 路径，loss 只读取 24 条训练录音的真实长度。测试用 greedy argmax 加 CTC collapse，并与最近原型基线使用同一个整句 exact match。测试噪声和时长模式未参与训练，但仍共享人工声学原型，结果只代表教学分布。

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.009)  # 创建更新手写 Conformer 与 CTC 头的 Adam。
training_history = []  # 保存 CTC loss、梯度和训练 exact match。
def greedy_decode_logits(logits, lengths):  # 把逐帧 logits 转成 CTC greedy 文本与路径。
    decoded = []  # 收集逐条最终文本。
    paths = []  # 收集逐帧 argmax 路径。
    for sample_index, length in enumerate(lengths.tolist()):  # 遍历录音并截断到真实帧。
        path = logits[sample_index, :length].argmax(dim=-1).tolist()  # 读取有效帧最大概率 token。
        paths.append(path)  # 保存原始路径便于教学检查。
        decoded.append(token_ids_to_text(ctc_collapse(path)))  # 应用 CTC collapse 得到命令文本。
    return decoded, paths  # 返回文本和原始路径。
for epoch in range(420):  # 在二十四条变长训练录音上执行真实训练。
    optimizer.zero_grad()  # 清除上一轮梯度。
    train_logits, train_attention, train_hidden = model(train_padded, train_valid_mask, strict_padding=True)  # 通过全层 mask 路径执行 Conformer 前向。
    loss, sample_losses = manual_ctc_loss(train_logits, train_targets, train_lengths)  # 用真实长度执行手写 CTC DP。
    loss.backward()  # 把路径总概率误差传播到 attention、卷积和声学投影。
    convolution_gradient = float(model.block.convolution.depthwise_weight.grad.norm().detach())  # 记录 depthwise kernel 的真实梯度。
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # 限制小型 CTC 训练中的极端梯度。
    optimizer.step()  # 更新全部手写模型参数。
    train_decoded, train_paths = greedy_decode_logits(train_logits.detach(), train_lengths)  # 读取当前训练 greedy 结果。
    train_exact = sum(prediction == name.split("-")[0] for prediction, name in zip(train_decoded, train_names)) / len(train_names)  # 计算二十四条训练整句准确率。
    training_history.append((float(loss.detach()), convolution_gradient, train_exact))  # 保存优化过程诊断。
with torch.no_grad():  # 关闭测试评估梯度。
    test_logits, test_attention, test_hidden = model(test_padded, test_valid_mask, strict_padding=True)  # 用严格 mask 路径编码六条留出录音。
test_decoded, test_paths = greedy_decode_logits(test_logits, test_lengths)  # 解码逐句文本和帧路径。
test_exact = sum(prediction == gold for prediction, gold in zip(test_decoded, test_names)) / len(test_names)  # 计算六句整句完全匹配率。
print("epoch  CTC loss  depthwise梯度  训练exact")  # 输出真实训练轨迹表头。
for epoch in (0, 49, 199, 419):  # 选择四个关键训练阶段。
    row = training_history[epoch]  # 读取当前阶段诊断。
    print(f"{epoch + 1:>4}   {row[0]:>7.4f}      {row[1]:>8.5f}      {row[2]:>6.1%}")  # 展示 CTC 收敛与卷积梯度。
print("命令  长度  Conformer帧路径       解码  正确  有效hidden范数")  # 输出逐句测试结果表头。
for sample_index, (name, length, path, prediction) in enumerate(zip(test_names, test_lengths.tolist(), test_paths, test_decoded)):  # 对齐六条测试结果。
    valid_hidden_norm = float(test_hidden[sample_index, :length].norm(dim=-1).mean())  # 计算有效帧平均编码范数。
    print(f"{name}    {length}    {path!s:<23}  {prediction:<4}  {prediction == name}    {valid_hidden_norm:.3f}")  # 展示原始路径、collapse 与编码强度。
print(f"整句 exact match：最近原型 Baseline={baseline_exact:.1%}，Conformer-CTC={test_exact:.1%}")  # 诚实汇总同口径结果。

epoch  CTC loss  depthwise梯度  训练exact
   1    8.0867       0.51207        0.0%
  50    0.0125       0.02908      100.0%
 200    0.0010       0.00046      100.0%
 420    0.0004       0.00010      100.0%
命令  长度  Conformer帧路径       解码  正确  有效hidden范数
开灯    5    [0, 1, 1, 3, 3]          开灯    True    6.562
关灯    6    [2, 2, 2, 3, 3, 3]       关灯    True    6.311
开窗    7    [0, 1, 1, 1, 4, 4, 0]    开窗    True    6.739
关窗    6    [0, 2, 2, 2, 0, 0]       关     False    6.751
开门    5    [1, 1, 5, 0, 5]          开门门   False    6.591
关门    7    [2, 2, 2, 5, 5, 5, 2]    关门关   False    6.311
整句 exact match：最近原型 Baseline=50.0%，Conformer-CTC=50.0%


## 结果解读

CTC loss 的下降、depthwise kernel 的非零梯度和逐帧路径共同证明模型真正参与了优化，但保存输出也显示训练 exact match 达到 100% 后，留出集仍只有 50%，与最近原型基线持平。“关窗”漏掉第二字，“开门/关门”因 blank 分隔产生重复字符，说明模型过拟合了有限时长模式，不能把训练拟合冒充语音泛化。Conformer 在这里的价值是展示全局 attention、局部卷积、无帧级对齐训练和严格 padding 合同。

## 失败案例：只 mask attention key，不清零 query 与卷积边界

下面把每条短录音的 padding 从零改成很大的任意数。严格实现会在输入投影后以及每个 residual/conv 边界清零，因此所有有效 logits 完全不变。故障对照复用同一组训练权重，但模拟“只 mask attention key、不清零 query/卷积输入”的旧路径；此时最后一个有效帧会经 kernel=3 读取第一个 padding 帧，从而发生可测漂移。

In [6]:
mutated_padding = test_padded.clone()  # 复制同一批真实声学输入供 padding 干预。
for sample_index, length in enumerate(test_lengths.tolist()):  # 遍历六条不同真实长度录音。
    if length < mutated_padding.shape[1]:  # 只修改确实存在的补齐帧。
        replacement = torch.linspace(15.0 + sample_index, 35.0 + sample_index, steps=(mutated_padding.shape[1] - length) * mutated_padding.shape[2])  # 生成与业务无关的大幅 padding 数值。
        mutated_padding[sample_index, length:] = replacement.reshape(mutated_padding.shape[1] - length, mutated_padding.shape[2])  # 覆盖 padding 而保持全部有效帧不变。
with torch.no_grad():  # 关闭不变性实验的梯度记录。
    strict_original_logits, strict_original_attention, strict_original_hidden = model(test_padded, test_valid_mask, strict_padding=True)  # 用正确路径处理零 padding。
    strict_mutated_logits, strict_mutated_attention, strict_mutated_hidden = model(mutated_padding, test_valid_mask, strict_padding=True)  # 用正确路径处理大数 padding。
    legacy_original_logits, legacy_original_attention, legacy_original_hidden = model(test_padded, test_valid_mask, strict_padding=False)  # 用故障路径处理零 padding。
    legacy_mutated_logits, legacy_mutated_attention, legacy_mutated_hidden = model(mutated_padding, test_valid_mask, strict_padding=False)  # 用故障路径处理大数 padding。
strict_differences = []  # 收集正确路径逐样本有效 logits 漂移。
legacy_differences = []  # 收集故障路径逐样本有效 logits 漂移。
boundary_differences = []  # 收集故障路径最后有效帧漂移。
for sample_index, length in enumerate(test_lengths.tolist()):  # 对每条录音只比较真实有效帧。
    strict_difference = float((strict_original_logits[sample_index, :length] - strict_mutated_logits[sample_index, :length]).abs().max())  # 计算严格路径有效输出最大差。
    legacy_difference = float((legacy_original_logits[sample_index, :length] - legacy_mutated_logits[sample_index, :length]).abs().max())  # 计算故障路径有效输出最大差。
    boundary_difference = float((legacy_original_logits[sample_index, length - 1] - legacy_mutated_logits[sample_index, length - 1]).abs().max())  # 检查紧邻 padding 的最后有效帧。
    strict_differences.append(strict_difference)  # 保存正确路径结果。
    legacy_differences.append(legacy_difference)  # 保存故障路径结果。
    boundary_differences.append(boundary_difference)  # 保存边界帧结果。
print("命令  长度  严格mask有效logit差  旧路径有效logit差  最后有效帧差")  # 输出 padding 干预实验表头。
for name, length, strict_difference, legacy_difference, boundary_difference in zip(test_names, test_lengths.tolist(), strict_differences, legacy_differences, boundary_differences):  # 对齐六条逐样本不变性结果。
    print(f"{name}    {length}       {strict_difference:.8f}          {legacy_difference:.6f}        {boundary_difference:.6f}")  # 展示正确路径不变和故障路径泄漏。
print(f"全体有效帧最大漂移：严格mask={max(strict_differences):.8f}，旧路径={max(legacy_differences):.6f}")  # 汇总修复前后的直接证据。

命令  长度  严格mask有效logit差  旧路径有效logit差  最后有效帧差
开灯    5       0.00000000          2.160779        2.160779
关灯    6       0.00000000          3.945007        3.945007
开窗    7       0.00000000          0.000000        0.000000
关窗    6       0.00000000          0.978456        0.978456
开门    5       0.00000000          1.859677        1.859677
关门    7       0.00000000          0.000000        0.000000
全体有效帧最大漂移：严格mask=0.00000000，旧路径=3.945007


## 生产差距与追问

真实 ASR 需要 log-Mel 特征、卷积下采样、多头相对位置 attention、多层 Conformer、SpecAugment、大词表与 beam search/语言模型融合。padding mask 还必须跟随下采样长度变化；若使用 BatchNorm，要避免 padding 参与统计。训练需监控 WER/CER、长音频显存、流式 chunk 与 look-ahead，数据切分应按说话人和设备完成。本实验只有合成六维原型，不能冒充真实语音识别泛化。

## 最小回归测试

In [7]:
assert len(test_names) >= 5  # 保证变长语音案例覆盖足够多命令。
assert float(test_hidden[~test_valid_mask].abs().max()) < 1e-7  # 保护严格 block 出口 padding 表示为零。
assert max(strict_differences) < 1e-7  # 保护任意改变 padding 数值都不影响有效 logits。
assert max(legacy_differences) > 1e-3  # 保护故障对照真实复现卷积边界污染。
assert training_history[-1][0] < training_history[0][0]  # 保护手写 CTC 训练确实降低负对数似然。
assert torch.isfinite(test_logits).all()  # 保护全部有效与 padding logits 保持有限数值。
print("最小回归测试通过：CTC 训练、逐层 padding mask 与有效 logits 不变性均保持有效。")  # 输出集中测试结论。

最小回归测试通过：CTC 训练、逐层 padding mask 与有效 logits 不变性均保持有效。
